## do_gp and iBME python script

Packages

In [ ]:
import subprocess
import os
import pandas as pd


Set Paths and Parameters

In [ ]:
#Working directory
print("Current working directory: {0}".format(os.getcwd()))
os.chdir('/home/malab/iBME')
print("Current working directory: {0}".format(os.getcwd()))

#Parameters for do_gp
path_structures = "/home/malab/Desktop/PDB_ID/Structures/Structures_test"
path_exp_file = "/home/malab/Desktop/BioEn-master/examples/scattering/files/experimental_data"
theta = "50" #theta
gl = "" #Optional- "" for loop
path_gpdoc = "/home/malab/iBME" #path to grid 

#Parameters for iBME
#will use exp_file
calc_rows_path = "/home/malab/iBME/GP{}/calc_rows.txt" #path to calc_rows.txt (will be GP(n))
#will use theta
out_name = "/home/malab/iBME/GP{}/"


Run 'do_gp.sh' and Pepsi SAXS

In [ ]:
#structure path, experiment path, theta, gl (optional), grid document
run = subprocess.run(["./do_gp_v3.sh", path_structures, "{}/SASDLU4.dat".format(path_exp_file),
                      theta, gl, "{}/GRID_tau".format(path_gpdoc)],
                      stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
print(f"STDOUT:\n {run.stdout}")
print(f"STDERR:\n {run.stderr}")

if run.returncode != 0:
    print(f"Script failed with return code {run.returncode}")
if run.returncode != 1:
    print(f"Script successful with return code {run.returncode}")


Create columns and files list necessary for iBME run

In [ ]:
#Counter file loop and make columns
counter = range(0, 10000) #Arbitrary end- increase if more directories necessary
files = 0
for i in counter:
    if os.path.isdir("/home/malab/iBME/GP{}".format(i)):
        files += 1
        # Read calc_saxs.txt with frame index + intensities
        calc_rows = pd.read_csv("/home/malab/iBME/GP{}/calc_saxs.txt".format(i),
                             header=None, delim_whitespace=True)

        # Drop the first column (frame index)
        calc_rows = calc_rows.drop(columns=[0])

        # Write calc_rows.txt with header line # DATA=SAXS
        calc_path = '/home/malab/iBME/GP{}/calc_rows.txt'.format(i)
        with open(calc_path, 'w') as f:
            f.write("# DATA=SAXS\n")
        calc_rows.to_csv(calc_path, mode='a', header=False, index=False, sep=' ')

    else:
        print(f"Amount of GP directories:")
